# WithoutSTandSRZeroShot 검증

`dp_agent_wrap_nost.py` 가 실제로 에이전트에 전달되는 프롬프트 `P` 에서
**ST(D)(결측 개수)와 SR(D)(원본 예시 행)** 를 제거했는지 확인한다.

| 기호 | 정보 | 이번 구성 |
|---|---|---|
| T0 | 스키마 정보 SCH(D) | 유지 |
| T1 | 카테고리 정보 CAT(D) | 유지 |
| T2 | 데이터 규모 정보 NR(D) | 프롬프트에 없음 (SR 블록 안에만 존재) |
| T3 | 통계 정보 ST(D) | **제거** |
| T4 | 원본 예시 정보 SR(D) | **제거** |

즉 `T_I(D)`, `I = {0, 1}`.

In [6]:
import os, sys

# 저장소 루트로 이동. 다른 위치에 두었다면 환경변수 TABULARQA_DIR 로 지정.
os.chdir(os.environ.get("TABULARQA_DIR", os.getcwd()))
sys.path.insert(0, os.getcwd())

print(sys.version)
print("CWD:", os.getcwd())

assert os.path.exists("core/prompt_generators.py"), "저장소 루트가 아닙니다"
assert os.path.exists("annotation/annotations_cot.json"), "annotation/ 이 없습니다"
assert os.path.exists("competition/000_CardBase/all.parquet"), "competition/000_CardBase 가 없습니다"
assert os.path.exists("dp_agent_wrap_no_ST_SR.py"), "dp_agent_wrap_no_ST_SR.py 가 같은 폴더에 있어야 합니다"
print("저장소 확인 완료")

3.12.13 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:26:47) [MSC v.1942 64 bit (AMD64)]
CWD: c:\Users\seude\Desktop\dp4agent
저장소 확인 완료


## 1. 대상 테이블 — 데이터에는 결측 정보가 존재한다

프롬프트에서 빠졌는지 확인하려면, 먼저 이 값이 **데이터에는 실제로 있다**는 것을 보여야 한다.

In [7]:
import importlib
import pandas as pd
import dp_agent_wrap_no_ST_SR as W
importlib.reload(W)

DS_NAME = "000_CardBase"

df = W.assert_local_table(DS_NAME)
print("shape:", df.shape)
print()
print("컬럼별 non-null 개수 (ST(D) 결측 정보에 해당):")
print(df.notnull().sum().to_string())
print()
print("컬럼별 null 개수:")
print(df.isnull().sum().to_string())

shape: (500, 4)

컬럼별 non-null 개수 (ST(D) 결측 정보에 해당):
Card_Number     500
Card_Family     500
Credit_Limit    500
Cust_ID         500

컬럼별 null 개수:
Card_Number     0
Card_Family     0
Credit_Limit    0
Cust_ID         0


## 2. CD 블록 비교 — ST 포함 vs ST 미포함

같은 테이블에 대해 두 생성기가 만드는 컬럼 설명을 나란히 출력한다.
`Non-Null CounT` 열이 사라진 것이 보여야 한다.

In [8]:
print(W.cd_block_diff(df))

--- ST 포함 (WithoutSRZeroShot, 기존 dp_agent_wrap_no_SR) ---
#,Column,Non-Null CounT,Dtype,Types of Elements,Values
0,Card_Number,500,str,[<class 'str'>],
1,Card_Family,500,str,[<class 'str'>],
2,Credit_Limit,500,int64,[<class 'int'>],
3,Cust_ID,500,str,[<class 'str'>],

--- ST 미포함 (WithoutSTandSRZeroShot, 이 파일) ---
#,Column,Dtype,Types of Elements,Values
0,Card_Number,str,[<class 'str'>],
1,Card_Family,str,[<class 'str'>],
2,Credit_Limit,int64,[<class 'int'>],
3,Cust_ID,str,[<class 'str'>],


## 3. 파이프라인 조립 및 최종 프롬프트 검사

API 키 없이 프롬프트만 확인하기 위해 더미 client 를 넣는다. LLM 호출은 하지 않는다.

In [9]:
class _DummyClient:
    """프롬프트 조립만 할 때 쓰는 자리표시자. 실제 호출은 하지 않는다."""
    pass


QUESTION = "What is the highest Credit_Limit?"
row = {"question": QUESTION, "dataset": DS_NAME}

pipe = W.build_pipe_nost(model="gpt-4o-mini", num_fix_attempts=2,
                         client=_DummyClient())

prompt = W.full_prompt(pipe, row)
print("최종 프롬프트 길이:", len(prompt), "chars")
print("few-shot 예시 수:", len(pipe.main_pipeline.prompt_generator.prompt_generator.shots))

Loading annotations from semeval train[:400]


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/49 [00:00<?, ?it/s]

최종 프롬프트 길이: 36528 chars
few-shot 예시 수: 9


## 4. 검증 — 프롬프트에 ST(D) / SR(D) 가 없어야 한다

In [10]:
info = W.assert_no_st(pipe, row)
print("CD 블록 수:", info["n_cd_blocks"], "(예시 9개 + 현재 질의 1개)")
print("CD 헤더   :", info["header"])
print("CD 컬럼 줄 합계:", info["n_columns"])

p = W.assert_no_sample_rows(pipe, row)
print()
print("SR 블록('rows from the dataframe') 부재 확인 완료")
print("'Non-Null' 등장 횟수:", p.count("Non-Null"))
print("'Non-Null CounT' 헤더 등장 횟수:", p.count(W.CD_HEADER_ORIGINAL))

CD 블록 수: 10 (예시 9개 + 현재 질의 1개)
CD 헤더   : #,Column,Dtype,Types of Elements,Values
CD 컬럼 줄 합계: 284

SR 블록('rows from the dataframe') 부재 확인 완료
'Non-Null' 등장 횟수: 0
'Non-Null CounT' 헤더 등장 횟수: 0


## 5. 블록 단위 전수 확인

CD 블록 하나하나의 헤더를 출력해, 예시 9개까지 전부 ST 가 빠졌는지 눈으로 확인한다.

In [11]:
blocks = W.find_cd_blocks(p)
for i, b in enumerate(blocks, 1):
    print(f"[block {i:02d}] 헤더={b[0]!r}  컬럼수={len(b)-1}")
    for line in b[1:]:
        print("          ", line)
    print()

[block 01] 헤더='#,Column,Dtype,Types of Elements,Values'  컬럼수=60
           0,ID<gx:number>,uint32,[<class 'int'>],
           1,Name<gx:text>,category,[<class 'str'>],5 example values are [' L. Suárez', ' K. De Bruyne', ' Bruno Fernandes', ' A. Griezmann', ' M. Acuña']
           2,Age<gx:number>,uint8,[<class 'int'>],
           3,Photo<gx:url>,category,[<class 'str'>],5 example values are ['https://cdn.sofifa.com/players/176/580/20_60.png', 'https://cdn.sofifa.com/players/192/985/20_60.png', 'https://cdn.sofifa.com/players/212/198/20_60.png', 'https://cdn.sofifa.com/players/194/765/20_60.png', 'https://cdn.sofifa.com/players/224/334/20_60.png']
           4,Nationality<gx:category>,category,[<class 'str'>],5 example values are ['Uruguay', 'Belgium', 'Portugal', 'France', 'Argentina']
           5,Overall<gx:number>,uint8,[<class 'int'>],
           6,Potential<gx:number>,uint8,[<class 'int'>],
           7,Club<gx:category>,category,[<class 'str'>, <class 'float'>],5 example values a

## 6. 최종 프롬프트 전문 확인

꼬리 부분(현재 질의 블록)을 출력하고, 전문을 파일로 저장한다.

In [12]:
print(prompt[-1800:])

with open("prompt_nost.txt", "w", encoding="utf-8") as f:
    f.write(prompt)
print()
print("저장: prompt_nost.txt")

Made,bool,[<class 'bool'>],
        11,gender,category,[<class 'str'>, <class 'float'>],All values are ['M', 'F']
        12,birthDate,datetime64[us, UTC],[<class 'pandas.Timestamp'>, <class 'pandas.api.typing.NaTType'>],
        13,title,category,[<class 'str'>, <class 'float'>],5 example values are ['CEO', 'Entrepreneur', 'Chairman and CEO', 'Cofounder', 'Cofounder and board member']
        14,philanthropyScore,float64,[<class 'float'>],
        15,bio,str,[<class 'str'>],
        16,about,str,[<class 'str'>, <class 'float'>],
    """


    df.columns = ['rank', 'personName', 'age', 'finalWorth', 'category', 'source', 'country', 'state', 'city', 'organization', 'selfMade', 'gender', 'birthDate', 'title', 'philanthropyScore', 'bio', 'about']
    
    # The columns used to answer the question: ['philanthropyScore', 'gender']
    # The types of the columns used to answer the question: ['number[UInt8]', 'category']
    # The type of the answer: category

    # Find the row with the high

## 7. (선택) 에러 수정 프롬프트도 같은 상태인지 확인

`ErrorFixingGeneratorClaudeVol2` 는 같은 `WithoutSTandSRZeroShot` 을 쓰므로
에러 수정 경로에서도 ST/SR 이 없어야 한다.

In [8]:
fix_row = (row, "    return None", "NameError: name 'x' is not defined")
fix_payload = pipe.error_fix_pipeline.prompt_generator(fix_row)
fix_text = W.payload_text(fix_payload)

print("에러 수정 프롬프트 길이:", len(fix_text))
print("'Non-Null' 등장 횟수:", fix_text.count("Non-Null"))
print("'rows from the dataframe' 등장 횟수:", fix_text.count("rows from the dataframe"))
assert "Non-Null" not in fix_text, "에러 수정 프롬프트에 ST 가 남아 있습니다"
assert "rows from the dataframe" not in fix_text, "에러 수정 프롬프트에 SR 이 남아 있습니다"
print("에러 수정 경로 검증 완료")

에러 수정 프롬프트 길이: 1430
'Non-Null' 등장 횟수: 0
'rows from the dataframe' 등장 횟수: 0
에러 수정 경로 검증 완료
